# Step 5c — RAG (Retrieval-Augmented Generation)

🇬🇧 **English** (this notebook)

RAG grounds a response in a document you provide: your text is chunked, embedded, and the most relevant chunks are retrieved and injected into the agent's context automatically. Unlike steps 3-5b, `knowledge_sources` does **not** work with a standalone `Agent.kickoff()` call — retrieval only gets wired up through a `Crew`'s own orchestration. So this one exercise needs a minimal inline `Crew` (still defined right here, no external YAML files) instead of a bare `Agent`.

## Background

> Lewis, P., Perez, E., Piktus, A., Petroni, F., Karpukhin, V., Goyal, N., Küttler, H., Lewis, M., Yih, W., Rocktäschel, T., Riedel, S., & Kiela, D. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*. NeurIPS 2020. [arXiv:2005.11401](https://arxiv.org/abs/2005.11401)

![RAG: query encoder and retriever feed top-k documents into a generator](../assets/rag-lewis2020-fig1.png)
*Figure 1 from Lewis et al. (2020). Reproduced for educational use in this course.*

**One practical detail**: embeddings use a separate model from the chat LLM. This notebook points the embedder at Gemini explicitly — you just need `GEMINI_API_KEY` set, even if `MODEL` is a different provider for chat.

## The exercise

Add a `.txt` file with background on your topic to `knowledge/`, then point `TextFileKnowledgeSource` at it. The one-agent, one-task `Crew` below is the minimum needed to activate retrieval — you're not editing any project files, everything is defined in this cell.

In [ ]:
import os

from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process
from crewai.knowledge.source.text_file_knowledge_source import TextFileKnowledgeSource

load_dotenv()

# ── Your knowledge document — add a .txt file to knowledge/ first ─────────────
knowledge_source = TextFileKnowledgeSource(file_paths=["TODO-your-document.txt"])

role      = "TODO: who is this agent?"
goal      = "TODO: what is this agent trying to achieve?"
backstory = "TODO: what expertise does this agent bring?"
agent = Agent(role=role, goal=goal, backstory=backstory, verbose=True)

question = "TODO: a question only answerable from your knowledge document"
task = Task(
    description=question,
    expected_output="A direct answer grounded in the provided knowledge.",
    agent=agent,
)

crew = Crew(
    agents=[agent],
    tasks=[task],
    process=Process.sequential,
    knowledge_sources=[knowledge_source],
    embedder={
        "provider": "google-generativeai",
        "config": {"api_key": os.getenv("GEMINI_API_KEY"), "model_name": "gemini-embedding-001"},
    },
)

result = crew.kickoff()
print(result.raw)

## Your task

1. Add a `.txt` file with background on your topic to `knowledge/`, and fill in the `TODO` fields (file name, agent identity, and a question only answerable from that document).

2. Run the cell. Does the agent answer correctly using the document?

3. Now comment out `knowledge_sources=[knowledge_source]` (pass an empty list instead) and run again. Does the agent still answer the document-specific question correctly, or does it admit it doesn't know? That comparison is the point of RAG.

4. Fill in the **Step 5c** section of `EVALUATION.md` and your final recommendation.

## Stretch goal

Add a PDF instead of a plain text file:
```python
from crewai.knowledge.source.pdf_knowledge_source import PDFKnowledgeSource
PDFKnowledgeSource(file_paths=["your_document.pdf"])
```
Ask a question whose answer appears on a specific page. Does the agent retrieve it?

---

**This is your final submission.** See [Assignment Overview](../../team_assignment/en/assignment-overview.md) for the full grading rubric and exactly what to submit.